# 🔬 Cell Tracking with `tracksdata` — A Detailed Tutorial

### Biohub – Cell Tracking During Development (Kaggle)

This notebook is a **hands-on, from-scratch walkthrough** of the data, the
`tracksdata` graph format, and — most importantly — **exactly how the competition
metric works**, verified with small worked examples you can read line by line.

It is built around the official baseline repo
[`royerlab/kaggle-cell-tracking-competition`](https://github.com/royerlab/kaggle-cell-tracking-competition).

---
### The problem

Biologists image developing **zebrafish embryos** with a microscope that captures a
full **3D volume** `(Z, Y, X)` at many **time points** `T`. To study how tissues form,
they need to know, for every cell: *where did it come from, where did it go, and when
did it divide?* Doing this by hand is impossibly slow — thousands of similar-looking
cells that move, deform, and split.

Your task: **detect cells in every frame and link them across time**, reconstructing
each cell's lineage (including divisions).

**The twist — sparse ground truth.** Only a *subset* of cells is annotated. Your method
must learn from sparse labels, and you are scored on a random sparse subset of cells.

---
### What this notebook covers

1. **Setup** — minimal install (metrics need *no* GPU / PyTorch)
2. **`tracksdata`** — the graph format: nodes, edges, divisions, attributes,
   plus **`networkx` visualization** and **graph properties** (lineages, degrees, depth)
3. **The data** — OME-Zarr 3D+time structure, reading a frame
4. **Submission format** — turning a graph into `submission.csv`
5. **The metric, step by step** — node matching (7 µm) → edge TP/FP/FN → Jaccard →
   node-count penalty → division Jaccard → final score
6. **Mock tests A–H** — every behaviour demonstrated and verified with `evaluate()`
7. **Takeaways**

## 1. Setup

The evaluation code (`tracking_cellmot.metrics`) depends only on `tracksdata` and
`polars` — **not** on PyTorch, CUDA, or zarr. So this tutorial stays light: we
`pip install tracksdata`, then add the repo's `src/` to the path so we can import
`tracking_cellmot.metrics` directly (no heavy `pip install -e .`).

In [ ]:
%matplotlib inline
import sys, os, subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'tracksdata', 'zarr', 'networkx', 'matplotlib'], check=True)

# Clone the baseline repo just to import its (pure-python) metric code.
if not os.path.exists('kaggle-cell-tracking-competition'):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/royerlab/kaggle-cell-tracking-competition'], check=True)
sys.path.insert(0, 'kaggle-cell-tracking-competition/src')

import numpy as np
import polars as pl
import networkx as nx
import matplotlib.pyplot as plt
import tracksdata as td
from tracking_cellmot.metrics import (
    evaluate, evaluate_datasets, per_sample_metrics, node_recall,
    ADJUSTMENT_ALPHA, SCORE_DIVISION_WEIGHT,
)

K = td.DEFAULT_ATTR_KEYS

# Silence two benign tracksdata messages so mock-test output stays clean:
#  - 'No matching edges found.' (expected when a case has 0 matchable edges, e.g. Mock D2)
#  - 'Graph already matched, overwriting previous matching.' (re-evaluating a graph)
import logging, warnings
logging.getLogger('tracksdata.utils._logging').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', message='Graph already matched.*')

print('tracksdata version :', td.__version__)
print('penalty alpha      :', ADJUSTMENT_ALPHA)
print('division weight    :', SCORE_DIVISION_WEIGHT)

### The attribute-key system

`tracksdata` stores attributes under **standard string keys**, exposed via
`td.DEFAULT_ATTR_KEYS` so code doesn't hard-code strings. The ones we use:

In [ ]:
print('time          K.T           =', repr(K.T))
print('coords        K.Z, K.Y, K.X =', repr(K.Z), repr(K.Y), repr(K.X))
print('node id       K.NODE_ID     =', repr(K.NODE_ID))
print('edge endpoints K.EDGE_SOURCE, K.EDGE_TARGET =', repr(K.EDGE_SOURCE), repr(K.EDGE_TARGET))

## 2. The `tracksdata` graph format

A tracking result is a **directed graph in space-time**:

| Element | Meaning | Stored as |
|---|---|---|
| **node** | one detected cell at time `t`, at centroid `(z, y, x)` | a vertex with attrs `t, z, y, x` |
| **edge** | the same cell (or its daughter) one frame later | a directed edge `source → target` |
| **division** | a cell splitting into two | a node with **two outgoing edges** (out-degree = 2) |

There is no separate 'division' object — a division is simply *detected* as
`out_degree(node) == 2`. Likewise a normal move is `out_degree == 1`, and a track
ending (or a cell leaving) is `out_degree == 0`.

### 2.1 Building a graph

We'll use `InMemoryGraph`. Note: `t` is a built-in attribute, but custom float
attributes (`z, y, x`) must be **declared** with `add_node_attr_key` before use.

In [ ]:
def build_graph(nodes, edges):
    """Build an InMemoryGraph.

    nodes : dict  name -> (t, z, y, x)
    edges : list  of (source_name, target_name)
    returns (graph, name->node_id map)
    """
    g = td.graph.InMemoryGraph()
    for key in (K.Z, K.Y, K.X):              # declare coordinate attributes
        g.add_node_attr_key(key, pl.Float64, 0.0)
    name2id = {}
    for name, (t, z, y, x) in nodes.items():
        name2id[name] = g.add_node({K.T: int(t), K.Z: float(z),
                                    K.Y: float(y), K.X: float(x)})
    for s, t in edges:
        g.add_edge(name2id[s], name2id[t], {})
    return g, name2id

# One cell moving over t=0,1,2 ; plus a parent at t=1 that divides into two at t=2
nodes = {
    'A0' : (0, 32, 100, 100),
    'A1' : (1, 32, 104, 100),    # moved +4 voxels in y
    'A2' : (2, 32, 108, 100),
    'P'  : (1, 32,  60,  60),    # a parent ...
    'Da' : (2, 32,  58,  64),    # ... daughter 1
    'Db' : (2, 32,  62,  56),    # ... daughter 2
}
edges = [('A0','A1'), ('A1','A2'), ('P','Da'), ('P','Db')]
g, ids = build_graph(nodes, edges)
print('num_nodes =', g.num_nodes(), ' num_edges =', g.num_edges())

### 2.2 Inspecting nodes

`node_attrs(attr_keys=[...])` returns a **polars DataFrame** — the natural way to
query a graph in bulk.

In [ ]:
df = g.node_attrs(attr_keys=[K.NODE_ID, K.T, K.Z, K.Y, K.X])
print(df)

### 2.3 Degrees & divisions

`out_degree` / `in_degree` take a list of node ids and return aligned counts.
A **division** is `out_degree == 2`.

In [ ]:
node_ids = g.node_ids()
out_deg = np.asarray(g.out_degree(node_ids))
in_deg  = np.asarray(g.in_degree(node_ids))
for nid, od, idg in zip(node_ids, out_deg, in_deg):
    role = 'DIVISION' if od == 2 else ('move' if od == 1 else 'track end')
    print(f'node {nid}: out={od} in={idg}  -> {role}')
print('\n# divisions =', int((out_deg == 2).sum()))

### 2.4 Visualizing a graph with `networkx`

A `tracksdata` graph maps cleanly onto a `networkx.DiGraph`, which is great for plotting
and for off-the-shelf graph algorithms. We convert by copying nodes (with their `t,z,y,x`
attributes) and directed edges.

To make the picture meaningful we first build a richer **lineage**: a root cell that
divides twice (a binary lineage tree) plus a second, independent track.

In [ ]:
def to_networkx(graph):
    G = nx.DiGraph()
    ndf = graph.node_attrs(attr_keys=[K.NODE_ID, K.T, K.Z, K.Y, K.X])
    for r in ndf.iter_rows(named=True):
        G.add_node(r[K.NODE_ID], t=r[K.T], z=r[K.Z], y=r[K.Y], x=r[K.X])
    edf = graph.edge_attrs(attr_keys=[K.EDGE_SOURCE, K.EDGE_TARGET])
    for r in edf.iter_rows(named=True):
        G.add_edge(r[K.EDGE_SOURCE], r[K.EDGE_TARGET])
    return G

# A lineage tree (root divides at t1 and again at t3) + an independent track.
lin_nodes = {
    'R0': (0, 32, 100, 100),
    'R1': (1, 32, 100, 100),                       # R1 divides ->
    'B2': (2, 32,  85, 100), 'C2': (2, 32, 115, 100),
    'B3': (3, 32,  85, 100), 'C3': (3, 32, 115, 100),   # C3 divides ->
    'C4a': (4, 32, 108, 100), 'C4b': (4, 32, 122, 100),
    'B4': (4, 32, 85, 100),
    # independent track
    'T0': (0, 32, 40, 40), 'T1': (1, 32, 44, 42), 'T2': (2, 32, 48, 44),
}
lin_edges = [('R0','R1'), ('R1','B2'), ('R1','C2'),
             ('B2','B3'), ('C2','C3'), ('B3','B4'),
             ('C3','C4a'), ('C3','C4b'),
             ('T0','T1'), ('T1','T2')]
lin, _ = build_graph(lin_nodes, lin_edges)
G = to_networkx(lin)
print('networkx DiGraph:', G.number_of_nodes(), 'nodes,', G.number_of_edges(), 'edges')

We lay nodes out as **time on the x-axis, vertical position on the y-axis**, and colour
dividing cells (out-degree 2) in red. This is the classic 'lineage tree' view.

In [ ]:
pos = {n: (d['t'], d['y']) for n, d in G.nodes(data=True)}
out_deg = dict(G.out_degree())
colors = ['#d62728' if out_deg[n] == 2 else '#1f77b4' for n in G.nodes()]

fig, ax = plt.subplots(figsize=(9, 5))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowsize=12,
                       edge_color='#888', width=1.5,
                       connectionstyle='arc3,rad=0.0')
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colors, node_size=600)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8, font_color='white')
ax.set_xlabel('time  t'); ax.set_ylabel('y centroid (voxel)')
ax.set_title('Lineage tree  (red = division, out-degree 2)')
ax.grid(alpha=0.2); plt.tight_layout(); plt.show()

### 2.5 Graph properties (and what they mean biologically)

Tracking graphs have structure worth quantifying. Each property maps to a biological
question:

| Graph property | Biological meaning |
|---|---|
| **weakly-connected components** | distinct cell **lineages** (a founder cell + all descendants) |
| **out-degree 0** | a track **ends** (cell dies, leaves the field of view, or annotation stops) |
| **out-degree 1** | a normal **move** to the next frame |
| **out-degree 2** | a **division** |
| **in-degree 0** | a track **start** (founder / cell enters) |
| **in-degree 2+** | a **merge** — physically impossible for cells, so a sign of a tracking error |
| **longest path** | the deepest **lineage depth** (generations captured) |

`networkx` computes all of these directly.

In [ ]:
comps = list(nx.weakly_connected_components(G))
print('lineages (weakly-connected components):', len(comps))
for i, c in enumerate(sorted(comps, key=len, reverse=True)):
    sub = G.subgraph(c)
    n_div = sum(1 for _, d in sub.out_degree() if d == 2)
    depth = nx.dag_longest_path_length(sub)
    print(f'  lineage {i}: {sub.number_of_nodes()} cells, {n_div} division(s), depth {depth}')

out_hist = np.bincount([d for _, d in G.out_degree()])
in_hist  = np.bincount([d for _, d in G.in_degree()])
print('\nout-degree counts:', {k: int(v) for k, v in enumerate(out_hist)})
print('in-degree  counts:', {k: int(v) for k, v in enumerate(in_hist)})
print('is a DAG (no cycles)?', nx.is_directed_acyclic_graph(G))
# in-degree >= 2 would be an (impossible) merge -> a tracking error flag
merges = [n for n, d in G.in_degree() if d >= 2]
print('illegal merges (in-degree>=2):', merges)

Visualize the degree distributions and per-lineage sizes:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
axes[0].bar(range(len(out_hist)), out_hist, color='#4C72B0')
axes[0].set_title('out-degree distribution'); axes[0].set_xlabel('out-degree'); axes[0].set_ylabel('# nodes')
axes[0].set_xticks(range(len(out_hist)))
axes[1].bar(range(len(in_hist)), in_hist, color='#DD8452')
axes[1].set_title('in-degree distribution'); axes[1].set_xlabel('in-degree')
axes[1].set_xticks(range(len(in_hist)))
sizes = sorted((len(c) for c in comps), reverse=True)
axes[2].bar(range(len(sizes)), sizes, color='#55A868')
axes[2].set_title('cells per lineage'); axes[2].set_xlabel('lineage'); axes[2].set_ylabel('# cells')
plt.tight_layout(); plt.show()

## 3. The competition data (3D + time)

Each video is an **OME-Zarr** store. Key facts:

- **Image array** lives at `<video>.zarr/0`, shape `(T, Z, Y, X)`, dtype `uint16`.
- **Chunking** is one full volume per timepoint: chunk shape `(1, Z, Y, X)`. Each chunk
  file is at `<video>.zarr/0/c/<t>/0/0/0` and is **blosc2-compressed**.
- **Voxel scale** `(Z, Y, X) = (1.625, 0.40625, 0.40625)` µm/voxel — anisotropic: Z is
  ~4× coarser than Y/X. *This matters for the metric, which measures distance in µm.*
- **Tracks** (training only) are sibling `<video>.geff` files: the sparse graph above.

Below we read the metadata of one **test** video (test videos have no `.geff`).

In [ ]:
import json, glob

TEST_DIR = '/kaggle/input/competitions/biohub-cell-tracking-during-development/test'
if not os.path.isdir(TEST_DIR):
    cands = glob.glob('/kaggle/input/*/test') + glob.glob('/kaggle/input/**/test', recursive=True)
    TEST_DIR = cands[0] if cands else TEST_DIR

zarrs = sorted(glob.glob(os.path.join(TEST_DIR, '*.zarr')))
print('TEST_DIR :', TEST_DIR)
print('# test videos found :', len(zarrs))

if zarrs:
    z0 = zarrs[0]
    arr_meta = json.load(open(os.path.join(z0, '0', 'zarr.json')))
    print('\nexample video :', os.path.basename(z0))
    print('  shape (T,Z,Y,X) :', arr_meta['shape'])
    print('  chunk shape     :', arr_meta['chunk_grid']['configuration']['chunk_shape'])
    print('  dtype           :', arr_meta['data_type'])

### 3.1 Reading one frame

Because each timepoint is its own blosc2 chunk, you can read a single frame cheaply
by decompressing one chunk file (this is what the nearest-neighbour baseline does).

In [ ]:
try:
    import blosc2
    if zarrs:
        shape = tuple(arr_meta['shape']); dtype = np.dtype(arr_meta['data_type'])
        chunk_path = os.path.join(z0, '0', 'c', '0', '0', '0', '0')   # t=0
        raw = blosc2.decompress(open(chunk_path, 'rb').read())
        vol = np.frombuffer(raw, dtype=dtype).reshape(shape[1:])      # (Z, Y, X)
        print('frame 0 volume :', vol.shape, vol.dtype)
        print('intensity  min/median/max :', vol.min(), int(np.median(vol)), vol.max())
except Exception as e:
    print('frame read skipped:', e)

## 4. Submission format

`submission.csv` is a flat table where **each row is either a `node` or an `edge`**:

```
id,dataset,row_type,node_id,t,z,y,x,source_id,target_id
0,vid1,node,1,0,32,128,128,-1,-1      # cell #1 at t=0
1,vid1,node,2,1,32,130,128,-1,-1      # cell #2 at t=1
2,vid1,edge,-1,-1,-1,-1,-1,1,2        # link cell 1 -> cell 2
```

Rules:
- `node` rows fill `node_id, t, z, y, x`; set `source_id = target_id = -1`.
- `edge` rows fill `source_id, target_id`; set the rest to `-1`.
- `node_id` restarts at 1 **per dataset**.
- A **division** is two `edge` rows sharing the same `source_id`.

Below: turn our graph `g` into submission rows programmatically.

In [ ]:
def graph_to_rows(graph, dataset):
    rows = []
    attrs = graph.node_attrs(attr_keys=[K.NODE_ID, K.T, K.Z, K.Y, K.X])
    for r in attrs.iter_rows(named=True):
        rows.append(dict(dataset=dataset, row_type='node', node_id=r[K.NODE_ID],
                         t=r[K.T], z=r[K.Z], y=r[K.Y], x=r[K.X],
                         source_id=-1, target_id=-1))
    edf = graph.edge_attrs(attr_keys=[K.EDGE_SOURCE, K.EDGE_TARGET])
    for r in edf.iter_rows(named=True):
        rows.append(dict(dataset=dataset, row_type='edge', node_id=-1,
                         t=-1, z=-1, y=-1, x=-1,
                         source_id=r[K.EDGE_SOURCE], target_id=r[K.EDGE_TARGET]))
    return rows

sub = pl.DataFrame(graph_to_rows(g, 'demo_video'))
print(sub)

## 5. The evaluation metric — step by step

This is the most important section. The scoring (see the repo's `metrics.md`) works in
six steps. Remember the GT is **sparse**, so the metric is careful to only judge what it
can.

### Step 1 — Node matching (≤ 7 µm)
Each predicted node is matched to at most one GT node by an **optimal bipartite
assignment** that minimises total centroid distance, with a hard cap of **7 µm**.
Distances are **physical**: `d = sqrt(Σ (Δvoxel · scale)²)`. A predicted node with no GT
node within 7 µm stays **unmatched** (and, crucially, is *not* punished as an FP — the GT
is sparse, so unmatched predictions are usually just unannotated real cells).

### Step 2 — Edge TP / FP / FN
For a predicted edge `u → v`:
- **TP**: `u` matches GT node `gu`, `v` matches GT node `gv`, **and** `gu → gv` is a real GT edge.
- **FP** (only in these two cases):
  - `v` matches a `gv` whose GT in-edge comes from a *different* source, or
  - `u` matches a `gu` whose GT out-edge goes to a *different* target.
- **FN**: a GT edge with no matching predicted edge.
- All other predicted edges are **ignored** (touching unannotated cells).

### Step 3 — Edge Jaccard
`J = TP / (TP + FP + FN)`.

### Step 4 — Adjusted edge Jaccard (node-count penalty)
To discourage flooding the volume with detections, the per-sample score is
`J_adj = max(0, J · (1 − α · node_ratio))`, with `α = 0.1` and
`node_ratio = (N_pred − N_target) / N_target`. More predicted nodes than expected ⇒
`node_ratio > 0` ⇒ the Jaccard is scaled **down**.

### Step 5 — Division Jaccard
Divisions get their own TP/FP/FN (was a real division correctly predicted as a division?),
giving `division_jaccard`.

### Step 6 — Final score
`score = adj_edge_jaccard + 0.1 · division_jaccard`.

Two helpers we'll use:
- `evaluate(pred, gt, scale)` → raw counts (`edge_tp/fp/fn`, `division_tp/fp/fn`, `num_pred_nodes`).
- `evaluate_datasets([...])` → micro-averaged **raw** edge Jaccard + division + score (a
  convenience that skips the node penalty). For the penalised number use
  `per_sample_metrics`, shown in Mock test F.

In [ ]:
SCALE = (1.625, 0.40625, 0.40625)   # Z, Y, X  microns / voxel

def phys_dist(a, b):
    """Physical distance (um) between two (z,y,x) voxel coords."""
    d = (np.array(a) - np.array(b)) * np.array(SCALE)
    return float(np.sqrt((d ** 2).sum()))

# sanity: how many voxels in y equal the 7 um cap?
print('1 voxel in y  =', round(phys_dist((0,0,0),(0,1,0)), 4), 'um')
print('10 voxels in y =', round(phys_dist((0,0,0),(0,10,0)), 3), 'um  (under 7 -> matches)')
print('20 voxels in y =', round(phys_dist((0,0,0),(0,20,0)), 3), 'um  (over 7  -> no match)')

def report(name, pred, gt, scale=SCALE):
    r = evaluate(pred, gt, scale=scale)
    print(f'--- {name} ---')
    print(f'  edges      TP/FP/FN = {r.edge_tp}/{r.edge_fp}/{r.edge_fn}')
    print(f'  divisions  TP/FP/FN = {r.division_tp}/{r.division_fp}/{r.division_fn}')
    denom = r.edge_tp + r.edge_fp + r.edge_fn
    j = r.edge_tp / denom if denom else float('nan')
    print(f'  edge Jaccard        = {j:.3f}   (num_pred_nodes={r.num_pred_nodes})')
    return r

> **Note.** Some mock tests below (e.g. **D2**, where a predicted node is shifted beyond
> 7 µm) are *designed* so that **zero** predicted edges can be matched — the metric
> correctly records 0 true positives. Internally `tracksdata` would log
> `No matching edges found.` for those cases; we raised its log level in the setup cell
> so the mock-test output stays clean. That message is expected and harmless, not an error.

## 6. Mock tests — verifying every behaviour

Shared ground truth: one cell tracked over `t = 0, 1, 2`.

In [ ]:
gt_nodes = {'A0': (0,32,100,100), 'A1': (1,32,104,100), 'A2': (2,32,108,100)}
gt_edges = [('A0','A1'), ('A1','A2')]
gt, _ = build_graph(gt_nodes, gt_edges)
print('GT: 3 nodes, edges A0->A1->A2')

### Mock A — Perfect prediction
Identical graph. Both GT edges are reproduced ⇒ `TP=2, FP=0, FN=0`, Jaccard = 1.0.

In [ ]:
pred, _ = build_graph(gt_nodes, gt_edges)
report('A. perfect', pred, gt)

### Mock B — A missing link (False Negative)
Correct nodes, but we forgot to link `A1 → A2`. That GT edge has no prediction ⇒ one **FN**.

In [ ]:
pred, _ = build_graph(gt_nodes, [('A0','A1')])
report('B. missing edge', pred, gt)   # expect TP=1, FN=1

### Mock C — An extra wrong link (False Positive)
We keep both correct edges *and* add `A0 → A2`. `A0`/`A2` match GT nodes, but `A0`'s real
GT out-edge goes to `A1`, not `A2` ⇒ that extra edge is an **FP**.

**Bonus subtlety:** `A0` now has out-degree 2, so the metric also sees a *spurious
division* ⇒ a division **FP** appears too. Great illustration of how edge mistakes can
create phantom divisions.

In [ ]:
pred, _ = build_graph(gt_nodes, [('A0','A1'), ('A1','A2'), ('A0','A2')])
report('C. extra wrong edge', pred, gt)   # expect edge FP=1, division FP=1

### Mock D — The 7 µm matching boundary
We displace the predicted `A1` in `y` and watch the match break. Sub-case D1 shifts by
10 voxels (≈ 4.06 µm, **under** 7 → still matches, edges survive). Sub-case D2 shifts by
20 voxels (≈ 8.12 µm, **over** 7 → `A1` unmatched, so *both* edges touching it collapse
to FN/FP).

In [ ]:
# D1: 10-voxel shift (~4.06 um) -- still within 7 um
d1 = dict(gt_nodes); d1['A1'] = (1, 32, 114, 100)
print('A1 shift =', round(phys_dist((32,104,100),(32,114,100)),2), 'um')
report('D1. shift 10vox (~4um, matches)', build_graph(d1, gt_edges)[0], gt)
print()
# D2: 20-voxel shift (~8.12 um) -- exceeds 7 um
d2 = dict(gt_nodes); d2['A1'] = (1, 32, 124, 100)
print('A1 shift =', round(phys_dist((32,104,100),(32,124,100)),2), 'um')
report('D2. shift 20vox (~8um, no match)', build_graph(d2, gt_edges)[0], gt)

### Mock E — Divisions: captured vs missed
New GT: parent `P` at t=0 divides into `Ca`, `Cb` at t=1. E1 predicts both daughter
links (division **TP**). E2 links only one daughter — a normal move, so the real
division is **missed** (division FN) and one daughter edge is an edge FN.

In [ ]:
gt_d_nodes = {'P': (0,32,60,60), 'Ca': (1,32,58,64), 'Cb': (1,32,62,56)}
gt_d_edges = [('P','Ca'), ('P','Cb')]
gt_d, _ = build_graph(gt_d_nodes, gt_d_edges)

report('E1. division captured', build_graph(gt_d_nodes, gt_d_edges)[0], gt_d)
print()
report('E2. division missed',  build_graph(gt_d_nodes, [('P','Ca')])[0], gt_d)

### Mock F — The node-count penalty (adjusted Jaccard)
`evaluate()` ignores predicted nodes that don't match GT, so spamming detections won't
hurt the *raw* Jaccard. The **adjusted** Jaccard does penalise it. We take a perfect
edge prediction but add many junk far-away nodes, then compute `per_sample_metrics`
against a target node count `n_total`.

In [ ]:
# perfect edges + lots of junk detections far from any GT node
many = dict(gt_nodes)
for i in range(20):
    many[f'junk{i}'] = (0, 10, 200 + i, 200)   # nowhere near GT
pred_many, _ = build_graph(many, gt_edges)

r = evaluate(pred_many, gt, scale=SCALE)
print('raw counts: edge TP/FP/FN =', r.edge_tp, r.edge_fp, r.edge_fn,
      '| num_pred_nodes =', r.num_pred_nodes)

n_total = 3            # the GT 'target' node count for this sample
rec = node_recall(pred_many, gt)
m = per_sample_metrics(r, n_total=n_total, node_recall=rec)
print(f"edge_jaccard      = {m['edge_jaccard']:.3f}  (raw, unpenalised)")
print(f"total_node_ratio  = {m['total_node_ratio']:.3f}  = (N_pred - N_target)/N_target")
print(f"adj_edge_jaccard  = {m['adj_edge_jaccard']:.3f}  = J*(1 - 0.1*node_ratio)")
print(f"node_recall       = {m['node_recall']:.3f}")

### Mock G — A partial result (fractional Jaccard)
A longer GT track of 4 edges where we get 3 right and add 1 wrong:
`J = 3 / (3 + 1 + 1) = 0.6`.

In [ ]:
g4_nodes = {f'N{t}': (t, 32, 100 + 4*t, 100) for t in range(5)}   # N0..N4
g4_edges = [(f'N{t}', f'N{t+1}') for t in range(4)]               # 4 GT edges
gt4, _ = build_graph(g4_nodes, g4_edges)

# predict 3 correct edges, drop one (N3->N4 => FN), add one wrong (N0->N2 => FP)
pred4, _ = build_graph(g4_nodes, [('N0','N1'), ('N1','N2'), ('N2','N3'), ('N0','N2')])
r = report('G. partial (expect J=0.6)', pred4, gt4)

### Mock H — Aggregating across datasets + final score
`evaluate_datasets` sums TP/FP/FN across all (pred, gt) pairs *before* computing the
Jaccard (micro-average), then `score = edge_jaccard + 0.1 · division_jaccard`.

In [ ]:
pairs = [
    (build_graph(gt_nodes, gt_edges)[0], gt),        # perfect edges
    (build_graph(gt_d_nodes, gt_d_edges)[0], gt_d),  # perfect division
    (pred4, gt4),                                    # the partial one from G
]
res = evaluate_datasets(pairs, scale=SCALE)
print('edge_jaccard     =', round(res.edge_jaccard, 4))
print('division_jaccard =', round(res.division_jaccard, 4))
print('FINAL score      =', round(res.score, 4),
      '  (= edge_jaccard + 0.1 * division_jaccard)')

## 7. Takeaways

- **Graph = nodes `(t,z,y,x)` + directed edges**; a **division is out-degree 2**. There is
  no separate division object.
- The data is **3D + time** and **sparsely annotated** — unmatched predictions are not
  punished as FPs, but the **node-count penalty** discourages mass over-detection.
- Matching is **physical (µm)** with a **7 µm cap** — because Z is 4× coarser, the same
  voxel error costs more in Z than in Y/X.
- Scoring pipeline: **match (7 µm) → edge TP/FP/FN → Jaccard → ×(node penalty) →
  + 0.1 · division Jaccard**.

**Strategy implications**
1. **Detection quality first.** An edge can only be a TP if *both* endpoints match GT
   nodes (≤ 7 µm). Poor localisation silently turns edges into FN/FP.
2. **Don't over-detect.** Extra nodes are fine for recall but the adjusted Jaccard scales
   you down via `node_ratio`.
3. **Divisions are worth chasing** — they add up to `0.1` to the score and are where
   naive nearest-neighbour linkers fail (a parent needs *two* out-edges).

### Next steps
- Generate a first `submission.csv` with the nearest-neighbour baseline (detect by
  thresholding + connected components, link by Hungarian assignment).
- Train the repo's `TemporalUNet3D` + `SimpleNodeTransformer` baseline for a learned
  detector + linker.